# Wellspring Clinics: Data Cleaning and Exploratory Analysis

**Author:** Odoh Ekenedirichukwu Johnpaul
**Tools:** Python (pandas, matplotlib, seaborn), then SQL, then Power BI

## Project overview
Wellspring runs six clinics in the Abuja area. This notebook cleans the raw workbook and explores visits, waiting times, follow-ups, revenue and patients. The clean tables are exported for SQL and Power BI.

## Business questions
1. How many visits does each clinic handle?
2. Which departments and diagnoses drive the most visits?
3. Where do patients wait the longest?
4. What share of visits need a follow-up?
5. Which clinics earn the most revenue?
6. Who are the patients?
7. How is work spread across staff?

## 1. Setup
Import the libraries and set the file paths. `pandas` handles the data, `matplotlib` and `seaborn` draw the charts, and `Path` keeps file paths clean.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW = Path("data/raw/wellspring-clinics.xlsx")
CLEAN = Path("data/clean")

sheets = pd.read_excel(RAW, sheet_name=None)
clinics  = sheets["Clinics"].copy()
patients = sheets["Patients"].copy()
staff    = sheets["Staff"].copy()
visits   = sheets["Visits"].copy()

print(clinics.shape, patients.shape, staff.shape, visits.shape)

(6, 5) (142, 7) (24, 6) (422, 10)


In [2]:
clinics.head()

,ClinicID,ClinicName,State,ClinicType,DateOpened
0,101,Wellspring Clinic - Garki,FCT,Primary Care,2019-03-01
1,102,Wellspring Clinic - Wuse,Abuja,Primary Care,2020-06-15
2,103,Wellspring Maternal Center,Federal Capital Territory,Maternal Health,2018-11-20
3,104,Wellspring Clinic - Kubwa,FCT,Primary Care,2021-01-10
4,105,Wellspring Clinic - Lugbe,Abuja,General,2022-04-05


In [3]:
patients.head()

,PatientID,FirstName,LastName,Gender,DateOfBirth,State,RegistrationDate
0,5001,Ifeoma,Okon,Male,1988-09-25,FCT,10/06/2025
1,5002,Femi,Ogundipe,MALE,1975-09-25,FCT,05/09/2024
2,5003,Amaka,Bello,FEMALE,1985-01-08,FCT,29/03/2023
3,5004,Ngozi,Abubakar,female,2015-09-06,Kogi,13/09/2024
4,5005,Blessing,Abubakar,female,1994-07-22,Niger,13/06/2024


In [4]:
patients.tail()

,PatientID,FirstName,LastName,Gender,DateOfBirth,State,RegistrationDate
137,5138,Segun,Bello,female,1999-05-21,Niger,13/05/2023
138,5139,Ngozi,Suleiman,FEMALE,1971-11-23,Nasarawa,02/12/2024
139,5140,Ngozi,Okon,Female,2001-05-06,Federal Capital Territory,23/02/2025
140,5006,Yusuf,Nwosu,Male,NaN,Nasarawa,08/10/2024
141,5041,Femi,Danjuma,FEMALE,2011-02-23,Federal Capital Territory,10/12/2023


In [5]:
staff.head()

,StaffID,StaffName,Role,ClinicID,YearsExperience,HireDate
0,1001,Amaka Suleiman,Nurse,106,8,2020-12-13
1,1002,Chidi Abubakar,Community Health Worker,102,3,2016-10-31
2,1003,Amaka Bello,Nurse,104,7,2020-11-05
3,1004,Amaka Abubakar,Pharmacist,105,7,2016-03-02
4,1005,Chiamaka Nwosu,Pharmacist,106,15,2016-11-05


In [6]:
staff.tail()

,StaffID,StaffName,Role,ClinicID,YearsExperience,HireDate
19,1020,Yusuf Adeyemi,Doctor,104,17,2017-12-02
20,1021,Chidi Adeyemi,Nurse,101,6,2015-10-28
21,1022,Ngozi Okon,Pharmacist,104,13,2016-10-09
22,1023,Hauwa Abubakar,Pharmacist,104,1,2015-10-29
23,1024,Aisha Suleiman,Nurse,106,11,2022-03-16


In [7]:
visits.head()

,VisitID,PatientID,ClinicID,StaffID,VisitDate,Department,Diagnosis,WaitTimeMinutes,FollowUpRequired,ConsultationFee
0,9001,5058,102,1016,2024-04-13,Pharmacy,Diabetes Follow-up,42.0,YES,"₦3,000"
1,9002,5043,106,1014,08/01/2025,Family Planning,Postnatal Review,33.0,No,NGN 3000
2,9003,5121,102,1017,04-17-2025,Lab,Immunization,45.0,yes,NGN 3000
3,9004,5065,101,1012,2024-04-15,Immunization,Hypertension,58.0,yes,"₦7,500"
4,9005,5057,104,1001,2025-02-28,Family Planning,Anemia,31.0,No,"₦5,000"


In [8]:
visits.tail()

,VisitID,PatientID,ClinicID,StaffID,VisitDate,Department,Diagnosis,WaitTimeMinutes,FollowUpRequired,ConsultationFee
417,9418,5105,103,1006,10/02/2025,General Outpatient,Hypertension,37.0,No,NGN 10000
418,9419,5135,103,1006,2024-08-31,Lab,Diabetes Follow-up,53.0,YES,15000
419,9420,5056,102,1007,08-Oct-2024,Family Planning,Typhoid,36.0,N,NGN 15000
420,9011,5047,102,1017,02-07-2024,General Outpatient,Malaria,28.0,N,5000
421,9201,5023,101,1012,13/03/2024,Immunization,Immunization,40.0,YES,"₦5,000"


## 3. Data audit (before cleaning)

In [9]:
visits.info()

<class 'pandas.DataFrame'>
RangeIndex: 422 entries, 0 to 421
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   VisitID           422 non-null    int64  
 1   PatientID         422 non-null    int64  
 2   ClinicID          422 non-null    int64  
 3   StaffID           422 non-null    int64  
 4   VisitDate         422 non-null    str    
 5   Department        422 non-null    str    
 6   Diagnosis         422 non-null    str    
 7   WaitTimeMinutes   404 non-null    float64
 8   FollowUpRequired  422 non-null    str    
 9   ConsultationFee   422 non-null    object 
dtypes: float64(1), int64(4), object(1), str(4)
memory usage: 48.3+ KB


In [10]:
visits.isna().sum()

VisitID              0
PatientID            0
ClinicID             0
StaffID              0
VisitDate            0
Department           0
Diagnosis            0
WaitTimeMinutes     18
FollowUpRequired     0
ConsultationFee      0
dtype: int64

In [11]:
visits.describe()

,VisitID,PatientID,ClinicID,StaffID,WaitTimeMinutes
count,422.000000,422.000000,422.000000,422.000000,404.000000
mean,9210.004739,5067.206161,103.495261,1011.668246,31.113861
std,121.488707,40.881587,1.701955,6.705115,21.707551
min,9001.000000,5001.000000,101.000000,1001.000000,-67.000000
25%,9105.250000,5031.000000,102.000000,1006.000000,23.000000
50%,9209.500000,5063.000000,103.000000,1012.000000,35.000000
75%,9314.750000,5103.750000,105.000000,1017.000000,43.000000
max,9420.000000,5140.000000,106.000000,1024.000000,93.000000


In [12]:
visits[visits["WaitTimeMinutes"]<0]

,VisitID,PatientID,ClinicID,StaffID,VisitDate,Department,Diagnosis,WaitTimeMinutes,FollowUpRequired,ConsultationFee
8,9009,5022,105,1003,2025-06-22,Family Planning,Respiratory Infection,-28.0,N,NGN 5000
40,9041,5060,101,1012,2025-04-28,Pharmacy,Routine Checkup,-46.0,yes,7500
70,9071,5020,106,1018,03/02/2024,Lab,Diabetes Follow-up,-40.0,yes,15000
72,9073,5026,104,1003,14-Dec-2024,Lab,Diabetes Follow-up,-34.0,No,"₦7,500"
74,9075,5098,106,1005,02-02-2025,Immunization,Hypertension,-4.0,No,NGN 5000
109,9110,5094,105,1010,2024-03-15,Pediatrics,Hypertension,-34.0,Yes,7500
127,9128,5110,102,1007,10-08-2024,Family Planning,Routine Checkup,-27.0,no,"₦15,000"
132,9133,5091,105,1004,05-19-2025,General Outpatient,Respiratory Infection,-19.0,no,NGN 2500
138,9139,5054,102,1016,2024-09-19,Pharmacy,Malaria,-47.0,YES,NGN 3000
150,9151,5104,102,1017,09-Feb-2024,Immunization,Antenatal Review,-39.0,Y,"₦5,000"


In [13]:
visits["WaitTimeMinutes"].value_counts()

WaitTimeMinutes
 40.0    19
 36.0    17
 38.0    15
 23.0    15
 28.0    13
         ..
-57.0     1
-33.0     1
-67.0     1
 3.0      1
-62.0     1
Name: count, Length: 80, dtype: int64

### 3.1 Missing values
Count the blank cells in each column.

In [14]:
staff.isna().sum()

StaffID            0
StaffName          0
Role               0
ClinicID           0
YearsExperience    0
HireDate           0
dtype: int64

**Finding:** The Staff table has no missing values in any column.
**Decision:** No action needed.

In [15]:
patients.isna().sum()

PatientID           0
FirstName           0
LastName            0
Gender              0
DateOfBirth         3
State               0
RegistrationDate    0
dtype: int64

**Finding:** `DateOfBirth` has 3 missing values. All other patient columns are complete.
**Decision:** Keep these patients. Their age will be blank, and they will be left out of age calculations only. Deleting them would remove their visits and distort the clinic totals.

In [16]:
visits.isna().sum()

VisitID              0
PatientID            0
ClinicID             0
StaffID              0
VisitDate            0
Department           0
Diagnosis            0
WaitTimeMinutes     18
FollowUpRequired     0
ConsultationFee      0
dtype: int64

**Finding:** `WaitTimeMinutes` has 18 missing values (about 4% of visits). All other visits columns are complete.
**Decision:** Leave them blank. I will not guess a waiting time. They will be ignored when calculating averages.

In [17]:
clinics.isna().sum()

ClinicID      0
ClinicName    0
State         0
ClinicType    0
DateOpened    0
dtype: int64

**Finding** Clinic has no Missing Values. All values are complete.
**Decision** No action neeeded

## 3.2 Checking for Duplicates

In [18]:
patients.duplicated().sum()

np.int64(2)

In [19]:
staff.duplicated().sum()

np.int64(0)

In [20]:
visits.duplicated().sum()

np.int64(2)

**Finding:** 2 patient rows and 2 visit rows are exact copies of another row, with the same ID and the same values in every column.
**Decision:** Remove the extra copies and keep one of each. A duplicated visit would be counted twice in visit totals and revenue, which would overstate both.

In [21]:
patients[patients.duplicated(keep=False)]

,PatientID,FirstName,LastName,Gender,DateOfBirth,State,RegistrationDate
5,5006,Yusuf,Nwosu,Male,NaN,Nasarawa,08/10/2024
40,5041,Femi,Danjuma,FEMALE,2011-02-23,Federal Capital Territory,10/12/2023
140,5006,Yusuf,Nwosu,Male,NaN,Nasarawa,08/10/2024
141,5041,Femi,Danjuma,FEMALE,2011-02-23,Federal Capital Territory,10/12/2023


**Finding:** Patients 5006 (Yusuf Nwosu) and 5041 (Femi Danjuma) each appear twice, with identical values in every column. The extra copies are at the bottom of the table (rows 140 and 141).
**Decision:** Remove the extra copies and keep one row per patient.

In [22]:
visits[visits.duplicated(keep=False)]

,VisitID,PatientID,ClinicID,StaffID,VisitDate,Department,Diagnosis,WaitTimeMinutes,FollowUpRequired,ConsultationFee
10,9011,5047,102,1017,02-07-2024,General Outpatient,Malaria,28.0,N,5000
200,9201,5023,101,1012,13/03/2024,Immunization,Immunization,40.0,YES,"₦5,000"
420,9011,5047,102,1017,02-07-2024,General Outpatient,Malaria,28.0,N,5000
421,9201,5023,101,1012,13/03/2024,Immunization,Immunization,40.0,YES,"₦5,000"


**Finding:** Visits 9011 and 9201 each appear twice, with identical values in every column. The extra copies are at the bottom of the table (rows 420 and 421).
**Decision:** Remove the extra copies and keep one row per visit. Left in, they would count the same visit twice and overstate visit totals and revenue.

In [23]:
visits["FollowUpRequired"].value_counts()

FollowUpRequired
No     63
 No    62
N      61
Y      61
yes    58
no     45
YES    38
Yes    34
Name: count, dtype: int64

**Finding:** `FollowUpRequired` has 8 spellings for 2 answers (Yes and No): `Y`, `yes`, `YES`, `Yes`, `N`, `no`, `No` and ` No` (with a leading space).
**Decision:** Remove extra spaces, then standardize everything to `Yes` or `No`. Otherwise a count by follow-up would split into 8 groups and give wrong percentages.

In [24]:
visits["ConsultationFee"].value_counts()

ConsultationFee
₦7,500       36
₦15,000      36
NGN 3000     32
₦10,000      29
₦5,000       28
NGN 5000     27
₦3,000       25
NGN 10000    25
₦2,500       25
NGN 15000    21
7500         20
NGN 7500     20
10000        19
5000         18
NGN 2500     18
15000        16
3000         16
2500         11
Name: count, dtype: int64

**Finding:** `ConsultationFee` is stored as text with 18 spellings for 6 real prices (2,500 / 3,000 / 5,000 / 7,500 / 10,000 / 15,000). Three styles are mixed: `₦` with commas, an `NGN` prefix, and plain numbers.
**Decision:** Remove the symbols, commas and `NGN`, then convert the column to numbers. As text, the fees cannot be added up, so revenue cannot be calculated.

In [25]:
patients["Gender"].value_counts()

Gender
female     24
FEMALE     18
F          16
 Female    15
male       14
Female     13
M          12
Male       11
Male       10
MALE        9
Name: count, dtype: int64

**Finding:** `Gender` has 10 spellings for 2 answers (Male and Female). The problems are mixed capitalization (`female`, `FEMALE`), single letters (`F`, `M`), and hidden extra spaces (` Female` and a `Male` with a trailing space).
**Decision:** Remove extra spaces, then standardize everything to `Male` or `Female`. Otherwise a gender chart in Power BI would show 10 bars instead of 2.

In [26]:
patients["State"].value_counts()

State
Nasarawa                     25
Federal Capital Territory    25
Niger                        24
Kogi                         22
FCT                          19
Kaduna                       14
Abuja                        13
Name: count, dtype: int64

**Finding:** `State` has 7 spellings for 5 real states. `Federal Capital Territory`, `FCT` and `Abuja` all mean the same place.
**Decision:** Standardize all three to `FCT`. The other four states are already consistent.

In [27]:
visits["WaitTimeMinutes"].describe()

count    404.000000
mean      31.113861
std       21.707551
min      -67.000000
25%       23.000000
50%       35.000000
75%       43.000000
max       93.000000
Name: WaitTimeMinutes, dtype: float64

**Finding:** `WaitTimeMinutes` has negative values (the minimum is -67). A waiting time cannot be negative, so these are recording errors. (Write the count here after you run the cell above.)
**Decision:** Set the negative values to missing. I will not turn -67 into 67 or guess a value, because the true wait is unknown. Missing values are ignored in averages.

In [28]:
visits[visits["WaitTimeMinutes"]<0]

,VisitID,PatientID,ClinicID,StaffID,VisitDate,Department,Diagnosis,WaitTimeMinutes,FollowUpRequired,ConsultationFee
8,9009,5022,105,1003,2025-06-22,Family Planning,Respiratory Infection,-28.0,N,NGN 5000
40,9041,5060,101,1012,2025-04-28,Pharmacy,Routine Checkup,-46.0,yes,7500
70,9071,5020,106,1018,03/02/2024,Lab,Diabetes Follow-up,-40.0,yes,15000
72,9073,5026,104,1003,14-Dec-2024,Lab,Diabetes Follow-up,-34.0,No,"₦7,500"
74,9075,5098,106,1005,02-02-2025,Immunization,Hypertension,-4.0,No,NGN 5000
109,9110,5094,105,1010,2024-03-15,Pediatrics,Hypertension,-34.0,Yes,7500
127,9128,5110,102,1007,10-08-2024,Family Planning,Routine Checkup,-27.0,no,"₦15,000"
132,9133,5091,105,1004,05-19-2025,General Outpatient,Respiratory Infection,-19.0,no,NGN 2500
138,9139,5054,102,1016,2024-09-19,Pharmacy,Malaria,-47.0,YES,NGN 3000
150,9151,5104,102,1017,09-Feb-2024,Immunization,Antenatal Review,-39.0,Y,"₦5,000"


**Finding:** 23 visits have a negative `WaitTimeMinutes` (the lowest is -67). A waiting time cannot be negative, so these are recording errors.
**Decision:** Set them to missing. I will not turn -28 into 28 or guess a value, because the true wait is unknown.

In [29]:
visits[visits.isna().any(axis=1)]

,VisitID,PatientID,ClinicID,StaffID,VisitDate,Department,Diagnosis,WaitTimeMinutes,FollowUpRequired,ConsultationFee
7,9008,5017,101,1021,17/01/2024,Pharmacy,Routine Checkup,NaN,N,NGN 3000
28,9029,5128,103,1006,06-Feb-2025,General Outpatient,Antenatal Review,NaN,Y,7500
54,9055,5029,101,1021,17/01/2024,General Outpatient,Hypertension,NaN,Yes,NGN 3000
156,9157,5069,106,1001,10-Jan-2024,General Outpatient,Routine Checkup,NaN,Y,NGN 10000
160,9161,5036,106,1014,10-29-2024,Family Planning,Immunization,NaN,no,NGN 5000
172,9173,5030,101,1011,26/09/2024,Immunization,Immunization,NaN,No,NGN 3000
189,9190,5030,103,1013,19-Oct-2024,Lab,Malaria,NaN,yes,NGN 2500
211,9212,5043,105,1004,02/07/2024,Pharmacy,Respiratory Infection,NaN,No,"₦3,000"
248,9249,5067,102,1002,19/06/2025,Immunization,Typhoid,NaN,No,"₦7,500"
251,9252,5029,106,1014,05-Oct-2024,General Outpatient,Diabetes Follow-up,NaN,Yes,NGN 3000


In [30]:
visits.isna().sum()[visits.isna().sum() > 0]

WaitTimeMinutes    18
dtype: int64

## 4. Data Cleaning              

### 4.1 ConsultationFee
The fee is stored as text with `₦`, `NGN` and commas. Remove them, then convert the column to numbers so the fees can be added up.

In [31]:
visits["ConsultationFee"]=visits["ConsultationFee"].astype(str)

In [32]:
visits["ConsultationFee"]=visits["ConsultationFee"].str.replace("₦", "", regex=False)

In [33]:
visits["ConsultationFee"]=visits["ConsultationFee"].str.replace("NGN", "", regex=False)

In [34]:
visits["ConsultationFee"] = visits["ConsultationFee"].str.replace(",", "", regex=False)

In [35]:
visits["ConsultationFee"] = visits["ConsultationFee"].str.strip()

In [36]:
visits["ConsultationFee"] = visits["ConsultationFee"].astype(int)

In [37]:
visits["ConsultationFee"].value_counts().sort_index()

ConsultationFee
2500     54
3000     73
5000     73
7500     76
10000    73
15000    73
Name: count, dtype: int64

In [38]:
visits["ConsultationFee"].dtype

dtype('int64')

In [39]:
visits.head()

,VisitID,PatientID,ClinicID,StaffID,VisitDate,Department,Diagnosis,WaitTimeMinutes,FollowUpRequired,ConsultationFee
0,9001,5058,102,1016,2024-04-13,Pharmacy,Diabetes Follow-up,42.0,YES,3000
1,9002,5043,106,1014,08/01/2025,Family Planning,Postnatal Review,33.0,No,3000
2,9003,5121,102,1017,04-17-2025,Lab,Immunization,45.0,yes,3000
3,9004,5065,101,1012,2024-04-15,Immunization,Hypertension,58.0,yes,7500
4,9005,5057,104,1001,2025-02-28,Family Planning,Anemia,31.0,No,5000


### 4.2 Duplicates
Remove the extra copies of 2 patients and 2 visits. Keep one row of each.

In [40]:
patients=patients.drop_duplicates()

In [41]:
patients.shape

(140, 7)

In [42]:
visits=visits.drop_duplicates()

In [43]:
visits.shape

(420, 10)

### 4.3 WaitTimeMinutes
A waiting time cannot be negative. Set the 23 negative values to missing. I will not guess the true wait.

In [44]:
visits.loc[visits["WaitTimeMinutes"]<0,"WaitTimeMinutes"] = np.nan

In [45]:
visits["WaitTimeMinutes"].isna().sum()

np.int64(41)

In [46]:
visits["WaitTimeMinutes"].describe()

count    379.000000
mean      35.192612
std       13.944802
min        3.000000
25%       25.000000
50%       36.000000
75%       44.000000
max       93.000000
Name: WaitTimeMinutes, dtype: float64

**Result:** The 23 negative waits are now missing. 41 of 420 visits (about 10%) have no valid wait time, so averages use the remaining 379 visits. The average wait is about 35 minutes.

### 4.4 VisitDate
The dates are text in four styles. I will separate the styles, then convert each one with its own format.

Label the four styles:
- slash: `13/03/2024`
- name: `14-Dec-2024`
- iso: `2025-06-22`
- dash: `05-19-2025`

formats = {
    "slash": ("%d/%m/%Y", slash),
    "name":  ("%d-%b-%Y", name),
    "iso":   ("%Y-%m-%d", iso),
    "dash":  ("%m-%d-%Y", dash),
}

parsed = pd.Series(pd.NaT, index=visits.index, dtype="datetime64[ns]")
for label, (fmt, mask) in formats.items():
    parsed.loc[mask] = pd.to_datetime(d[mask], format=fmt)

visits["VisitDate"] = parsed
visits["VisitDate"].isna().sum()

In [47]:
d = visits["VisitDate"].astype(str).str.strip()

In [48]:
slash=d.str.contains("/")

In [49]:
name=d.str.contains("[A-Za-z]")

In [50]:
iso = d.str[4] == "-"

In [51]:
dash = ~(slash | name | iso)

In [52]:
print(iso.sum(), slash.sum(), dash.sum(), name.sum())

114 106 100 100


In [53]:
parsed = pd.Series(pd.NaT, index=visits.index, dtype="datetime64[ns]")
parsed.loc[slash] = pd.to_datetime(d[slash], format="%d/%m/%Y")
parsed.loc[name]  = pd.to_datetime(d[name], format="%d-%b-%Y")
parsed.loc[iso]   = pd.to_datetime(d[iso], format="%Y-%m-%d")
parsed.loc[dash]  = pd.to_datetime(d[dash], format="%m-%d-%Y")

visits["VisitDate"] = parsed
visits["VisitDate"].isna().sum()

np.int64(0)

In [54]:
visits["VisitDate"].dtype

dtype('<M8[ns]')

In [55]:
visits[["VisitID","VisitDate"]].sort_values("VisitDate").head()

,VisitID,VisitDate
99,9100,2024-01-03
342,9343,2024-01-04
229,9230,2024-01-05
234,9235,2024-01-07
45,9046,2024-01-07


### 4.5 FollowUpRequired
`FollowUpRequired` has 8 spellings for 2 answers. Remove extra spaces, then standardize everything to `Yes` or `No`.

In [56]:
visits["FollowUpRequired"]=visits["FollowUpRequired"].str.strip().str.capitalize()

In [57]:
visits["FollowUpRequired"].value_counts()

FollowUpRequired
No     170
Yes    129
Y       61
N       60
Name: count, dtype: int64

In [58]:
visits["FollowUpRequired"]=visits["FollowUpRequired"].str.strip().str.lower().map({"yes":"Yes", "y":"Yes", "no":"No", "n":"No"})

In [59]:
visits["FollowUpRequired"].value_counts()

FollowUpRequired
No     230
Yes    190
Name: count, dtype: int64

In [60]:
visits["FollowUpRequired"].dtype

<StringDtype(na_value=nan)>

In [61]:
visits["FollowUpRequired"].value_counts()

FollowUpRequired
No     230
Yes    190
Name: count, dtype: int64

In [62]:
visits["FollowUpRequired"] = visits["FollowUpRequired"].str.strip().str.lower().map({"yes":"Yes", "y":"Yes", "no":"No", "n":"No"})

In [63]:
fresh = pd.read_excel(RAW, sheet_name="Visits")
visits["FollowUpRequired"] = fresh["FollowUpRequired"].str.strip().str.lower().map({"yes":"Yes", "y":"Yes", "no":"No", "n":"No"})
visits["FollowUpRequired"].value_counts()

FollowUpRequired
No     230
Yes    190
Name: count, dtype: int64

## 4.6 Patients  has 8 spelling for two answers(male and female)

In [64]:
fresh_patients = pd.read_excel(RAW, sheet_name="Patients")
patients["Gender"] = fresh_patients["Gender"].str.strip().str.lower().map({"male":"Male", "m":"Male", "female":"Female", "f":"Female"})
patients["Gender"].value_counts()

Gender
Female    85
Male      55
Name: count, dtype: int64

In [65]:
patients["State"] = fresh_patients["State"].str.strip().replace({"Federal Capital Territory": "FCT", "Abuja": "FCT"})
patients["State"].value_counts()

State
FCT         56
Niger       24
Nasarawa    24
Kogi        22
Kaduna      14
Name: count, dtype: int64

## 4.7 Clinics table has state name inconsistencies

In [66]:
fresh_Clinics = pd.read_excel(RAW, sheet_name="Clinics")
Clinics["State"] = fresh_Clinics["State"].str.strip().replace({"Federal Capital Territory": "FCT", "Abuja":"FCT"})
Clinics["State"].value_counts()

NameError: name 'Clinics' is not defined

### 5.0 Exporting the Data into the CSV

### 1: Create the folder

In [ ]:
CLEAN.mkdir(parents=True, exist_ok=True)

### 2: Export each table

In [ ]:
clinics.to_csv(CLEAN / "clinics_clean.csv", index=False)
patients.to_csv(CLEAN / "patients_clean.csv", index=False)
staff.to_csv(CLEAN / "staff_clean.csv", index=False)
visits.to_csv(CLEAN / "visits_clean.csv", index=False)

## 3: Confirm the files exist and check their shape

In [ ]:
import os
for name in ["clinics_clean.csv", "patients_clean.csv", "staff_clean.csv", "visits_clean.csv"]:
    path = CLEAN / name
    print(name, "exists:", path.exists())

In [ ]:
print("clinics:", clinics.shape)
print("patients:", patients.shape)
print("staff:", staff.shape)
print("visits:", visits.shape)